# Individual T-maze Thorough Report

This version retains all original report outputs:

1. Overall performance per ITI bin
2. 3-day moving-average performance
3. Attempts per bin and total attempts
4. Left/right side performance
5. Performance versus ITI
6. Raw exponential memory-decay fit
7. Smoothed exponential memory-decay fit
8. Task-event table

It also:

- restricts all calculations to the AC period from `**/report_output/tasks/kage*_tasks.csv`
- uses the requested ITI bins up to 10 minutes plus a 10+ minute bin
- adds a learning criterion of >60% for 3 consecutive days
- adds individual criterion annotations and a cohort-level learning summary page

9. Mouse-level CSV exports for group analysis
10. Cohort-level longitudinal mean ± SEM plots
11. Cohort-level performance-versus-ITI mean ± SEM plot


## 1. Imports

In [6]:
# This script will create the report for an individual kage
import os
import numpy as np
import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy.orm import sessionmaker
from datetime import datetime
from collections import Counter
from typing import Optional, List, Dict, Union
from scipy.optimize import curve_fit
from sklearn.metrics import r2_score
import matplotlib
import os
import re

matplotlib.use("Agg")  # Use non-interactive backend - must be before pyplot import
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import traceback
from pathlib import Path


## 2. Cohort configuration

In [7]:
kg = 1
homef = "/Volumes/Aimi_SmrtKg/2024-05-may-jul_NLGF6m"
db_path = os.path.join(homef, "DB")
kgs = [
    d
    for d in os.listdir(db_path)
    if os.path.isdir(os.path.join(db_path, d)) and d.startswith("kage")
]
kgs.sort(key=lambda x: int(re.search(r"\d+", x).group()))

print(kgs)

# kgs = ["kage1", "kage2", "kage3", "kage4", "kage5"]
kage = kgs[0]

['kage1', 'kage2', 'kage3', 'kage4', 'kage5', 'kage6', 'kage7', 'kage8', 'kage9', 'kage10', 'kage11', 'kage12', 'kage13', 'kage14', 'kage15', 'kage16', 'kage17', 'kage18', 'kage19', 'kage20', 'kage21', 'kage22', 'kage23', 'kage24', 'kage25', 'kage26', 'kage27', 'kage28', 'kage29', 'kage30', 'kage31', 'kage32', 'kage33']


## 3. Analysis and plotting functions

In [8]:
def bin_data_overlapping_windows(AC_subset):
    """
    Bin data using overlapping windows; bin position = mean dt within each bin.
    """

    windows = [
        (0, 120),  # 0–2 min
        (60, 180),  # 1–3 min
        (120, 300),  # 2–5 min
        (180, 360),  # 3–6 min
        (240, 420),  # 4–7 min
        (300, 540),  # 5–9 min
        (360, 600),  # 6–10 min
    ]

    dt = AC_subset["dt between ROI"].to_numpy()
    choice = AC_subset["choice"].to_numpy()

    # Pre-filter correct trials
    dt_correct = dt[choice == 1]

    n_windows = len(windows)
    binned_data = np.zeros(n_windows, dtype=int)
    binned_data_filtered = np.zeros(n_windows, dtype=int)
    percentage = np.zeros(n_windows, dtype=float)
    bin_positions = np.zeros(n_windows, dtype=float)

    for i, (start, end) in enumerate(windows):
        # All events in window
        mask_all = (dt >= start) & (dt < end)
        vals = dt[mask_all]

        # Correct events
        mask_correct = (dt_correct >= start) & (dt_correct < end)

        binned_data[i] = mask_all.sum()
        binned_data_filtered[i] = mask_correct.sum()

        # Percentage correct
        if binned_data[i] > 0:
            percentage[i] = 100 * (binned_data_filtered[i] / binned_data[i])
        else:
            percentage[i] = np.nan

        # Bin position = mean dt inside window (converted to minutes)
        bin_positions[i] = vals.mean() / 60 if len(vals) > 0 else np.nan

    # Overall mean dt
    x_prct_smoo = float(np.mean(dt)) if len(dt) > 0 else np.nan

    return {
        "binned_data": binned_data,
        "binned_data_filtered": binned_data_filtered,
        "percentage": percentage,
        "x_prct_smoo": x_prct_smoo,
        "bin_labels": bin_positions,  # <-- UPDATED
    }


def process_tmaze_file(homef, kage):
    tmaze_filename = os.path.join(
        homef, f"DB/{kage}/analysis/tmaze_output/{kage}_tmaze.csv"
    )
    if not os.path.exists(tmaze_filename):
        print(f"Missing file for {kage}, skipping.")
        return None

    df = pd.read_csv(tmaze_filename)
    if df.empty:
        print(f"No data for {kage}, skipping.")
        return None

    # Minimal filtering and prep
    required_cols = {"date", "dt between ROI", "choice", "side"}
    if not required_cols.issubset(df.columns):
        print(f"Columns missing for {kage}, skipping.")
        print(f"Available columns: {list(df.columns)}")
        return None

    dcopy = df.copy().dropna(subset=["date", "dt between ROI"])
    if dcopy.empty:
        print(f"No valid rows for {kage}, skipping.")
        return None

    return df, dcopy


def plot_tmaze_performance_arrays_old(
    all_performance_arr,
    moving_averages_arr,
    custom_bins=None,
    bin_labels=None,
    kagen="None",
    all_attempts_arr=None,
    daily_side_counts=None,
    daily_side_dates=None,
):
    """
    Plot four subplots: all_performance_arr, moving_averages_arr, attempts, and daily_side_counts.
    The attempts subplot displays one line per bin and a sum line in black.
    The fourth subplot visualizes daily_side_counts per day if provided.

    Parameters
    ----------
    all_performance_arr : np.ndarray or similar
        Performance data over days (possibly 2D: [bin, day]).
    moving_averages_arr : np.ndarray or dict or similar
        Moving average performance data (possibly 2D or dict of arrays).
    custom_bins : sequence, optional
        Custom bin edges for labeling.
    bin_labels : sequence of str, optional
        Bin labels for legend. If not provided, will be guessed from custom_bins or size.
    kagen : str
        Figure title, usually kage identifier.
    all_attempts_arr : np.ndarray or None, optional
        Number of attempts per bin per day, shape should match all_performance_arr if available.
    daily_side_counts : dict (optional)
        Dict {"left": {"correct": [...], "attempts": [...]}, "right": {...}} with per-day arrays.
    daily_side_dates : sequence (optional)
        Dates aligning with daily_side_counts arrays.
    """
    import matplotlib.pyplot as plt
    import numpy as np
    import itertools

    def get_x_length(arr):
        if isinstance(arr, dict):
            return max(
                a.squeeze().shape[-1] if hasattr(a, "squeeze") else len(a)
                for a in arr.values()
            )
        elif isinstance(arr, np.ndarray):
            if arr.ndim > 1:
                return arr.shape[1]
            else:
                return arr.shape[0]
        else:
            try:
                return len(arr)
            except Exception:
                return 0

    def moving_average(a, n=3):
        """Compute centered n-day moving average for 1D numpy array-like (n odd)."""
        a = np.asarray(a)
        if a.ndim == 0:
            return a
        if len(a) < n:
            return np.full_like(a, np.nan)
        ret = np.convolve(a, np.ones(n) / n, mode="valid")
        pad_len = (len(a) - len(ret)) // 2
        ret = np.pad(ret, (pad_len, len(a) - len(ret) - pad_len), mode="edge")
        return ret

    if daily_side_counts is not None:
        # print(f'daily_side_counts: {daily_side_counts}')
        if isinstance(daily_side_counts, dict):
            # print(f"  received daily_side_counts (keys: {list(daily_side_counts.keys())})")
            for side, v in daily_side_counts.items():
                for key in ("correct", "attempts"):
                    arr = v.get(key)
                    if arr is not None:
                        # print(f"    {side} {key} shape: {np.shape(arr)}")
                        if isinstance(arr, (list, np.ndarray)) and len(arr) > 0:
                            preview = np.asarray(arr).flatten()[:5]
                            print(f"      sample: {preview}")
                        else:
                            print(f"      (empty or not list/array)")
                    else:
                        print(f"    {side} {key} not found")

        else:
            print(f"  received daily_side_counts of type: {type(daily_side_counts)}")
            try:
                arr_shape = np.shape(daily_side_counts)
                # print(f"    shape: {arr_shape}")
                arr_flat = np.asarray(daily_side_counts).flatten()
                # print(f"    sample: {arr_flat[:5]}")
            except Exception as e:
                print(f"    Could not determine shape/sample: {e}")
    else:
        print(f"  NO daily_side_counts provided")
    if daily_side_dates is not None:
        print(
            f"  received daily_side_dates (length: {len(daily_side_dates) if hasattr(daily_side_dates,'__len__') else 'N/A'})"
        )
    else:
        print(f"  NO daily_side_dates provided")

    # Guess bin_labels if not supplied
    if bin_labels is None:
        if custom_bins is not None:
            try:
                bin_labels = [
                    f"{custom_bins[i]}-{custom_bins[i+1]}s"
                    for i in range(len(custom_bins) - 1)
                ]
            except Exception:
                bin_labels = None
        if bin_labels is None:
            if isinstance(all_performance_arr, np.ndarray):
                bin_labels = [f"Bin {i+1}" for i in range(all_performance_arr.shape[0])]
            else:
                bin_labels = [f"Bin 1"]

    n_subplot = 4

    x_len_all = get_x_length(all_performance_arr)
    x_len_mov = get_x_length(moving_averages_arr)
    x_len_att = (
        get_x_length(all_attempts_arr) if all_attempts_arr is not None else x_len_all
    )

    fig, axs = plt.subplots(n_subplot, 1, figsize=(12, 16), sharex=True)
    fig.suptitle(f"{kagen}", fontsize=14, weight="bold")

    # 1) all_performance_arr
    print("  Plotting Overal performance...")
    axs[0].set_title("Overal performance")
    if isinstance(all_performance_arr, np.ndarray):
        n_bins = all_performance_arr.shape[0]
        for i in range(n_bins):
            x = (
                np.arange(all_performance_arr.shape[1])
                if all_performance_arr.ndim > 1
                else np.arange(all_performance_arr.shape[0])
            )
            y = (
                all_performance_arr[i]
                if all_performance_arr.ndim > 1
                else all_performance_arr
            )
            axs[0].plot(
                x,
                y * 100,
                label=bin_labels[i] if i < len(bin_labels) else f"Bin {i+1}",
                linewidth=1.5,
            )  # 50% thicker
        axs[0].set_ylabel("Performance (%)")
        axs[0].legend(title="Bins")
    else:
        x = np.arange(len(all_performance_arr))
        axs[0].plot(
            x, np.array(all_performance_arr) * 100, linestyle="--", linewidth=0.7
        )  # 50% thicker
        axs[0].set_ylabel("Performance (%)")
    axs[0].set_xlabel("Day index")
    axs[0].set_ylim(0, 105)
    axs[0].grid(True, which="both")
    for yval in range(10, 100, 10):
        axs[0].axhline(y=yval, color="gray", alpha=0.3, linewidth=1.5)
    axs[0].set_xlim(0, x_len_all - 1 if x_len_all > 0 else 0)

    # 2) moving_averages_arr
    print("Plotting  T-Maze performance...")
    axs[1].set_title("T-Maze performance...")
    lines_color_cycle = axs[1]._get_lines.prop_cycler
    colors = [
        d.get("color", f"C{i}")
        for i, d in enumerate(itertools.islice(lines_color_cycle, 32))
    ]
    if isinstance(moving_averages_arr, dict):
        for i, (k, arr) in enumerate(moving_averages_arr.items()):
            y = arr.squeeze() if hasattr(arr, "squeeze") else np.array(arr)
            x = np.arange(y.shape[-1]) if y.ndim > 0 else np.arange(1)
            color = colors[i % len(colors)]
            line_label = bin_labels[i] if i < len(bin_labels) else str(k)
            axs[1].plot(
                x, y * 100, linestyle="--", label=line_label, color=color, linewidth=1.5
            )  # now dashed & thicker
            smoothed = moving_average(y, n=3) * 100
            axs[1].plot(
                x, smoothed, linestyle="solid", color=color, alpha=0.7, linewidth=2.25
            )  # now solid & 50% thicker than above
        axs[1].legend(title="Bins")
    elif isinstance(moving_averages_arr, np.ndarray):
        n_bins = moving_averages_arr.shape[0]
        for i in range(n_bins):
            x = (
                np.arange(moving_averages_arr.shape[1])
                if moving_averages_arr.ndim > 1
                else np.arange(moving_averages_arr.shape[0])
            )
            y = (
                moving_averages_arr[i]
                if moving_averages_arr.ndim > 1
                else moving_averages_arr
            )
            color = colors[i % len(colors)]
            line_label = bin_labels[i] if i < len(bin_labels) else f"Bin {i+1}"
            axs[1].plot(
                x, y * 100, linestyle="--", label=line_label, color=color, linewidth=1.5
            )  # now dashed & thicker
            smoothed = moving_average(y, n=3) * 100
            axs[1].plot(
                x, smoothed, linestyle="solid", color=color, alpha=0.7, linewidth=2.25
            )  # now solid & 50% thicker than above
        axs[1].legend(title="Bins")
    else:
        x = np.arange(len(moving_averages_arr))
        y = np.array(moving_averages_arr)
        axs[1].plot(x, y * 100, linestyle="--", linewidth=1.5)
        smoothed = moving_average(y, n=3) * 100
        axs[1].plot(
            x, smoothed, linestyle="solid", color="C0", alpha=0.7, linewidth=2.25
        )
    axs[1].set_xlabel("Day index")
    axs[1].set_ylabel("Performance (%)")
    axs[1].set_ylim(0, 105)
    axs[1].grid(True, which="both")
    for yval in range(10, 100, 10):
        axs[1].axhline(y=yval, color="gray", alpha=0.3, linestyle="--", linewidth=0.7)
    axs[1].set_xlim(
        0, max(x_len_mov, x_len_all) - 1 if max(x_len_mov, x_len_all) > 0 else 0
    )

    # 3) attempts per bin line + total
    print("  Drinking attempts...")
    axs[2].set_title("Attempts per bin (and total)")
    n_bins = 1
    att_shape = None
    if all_attempts_arr is not None:
        att_arr = np.array(all_attempts_arr)
        att_shape = att_arr.shape
        if att_arr.ndim == 1:
            n_bins = 1
            att_per_bin = [att_arr]
            time_x = np.arange(att_arr.shape[0])
        elif att_arr.ndim == 2:  # (bins, days)
            n_bins = att_arr.shape[0]
            att_per_bin = [att_arr[i] for i in range(n_bins)]
            time_x = np.arange(att_arr.shape[1])
        else:
            time_x = np.arange(get_x_length(att_arr))
            att_per_bin = [att_arr]
        for i in range(n_bins):
            color = f"C{i % 10}"
            lbl = bin_labels[i] if i < len(bin_labels) else f"Bin {i+1}"
            y = np.asarray(att_per_bin[i])
            axs[2].plot(time_x, y, color=color, label=lbl, linewidth=1.5)  # 50% thicker
        total_attempts = np.sum(att_arr, axis=0) if att_arr.ndim > 1 else att_arr
        axs[2].plot(
            time_x, total_attempts, color="black", lw=3, label="sum", zorder=4
        )  # 50% thicker than previous lw=2
        axs[2].legend(title="Bins")
    else:
        axs[2].text(
            0.5,
            0.5,
            "No attempt data",
            ha="center",
            va="center",
            transform=axs[2].transAxes,
        )
        axs[2].set_xlim(0, x_len_att - 1 if x_len_att > 0 else 0)
    axs[2].set_xlabel("Day index")
    axs[2].set_ylabel("Attempts")
    axs[2].grid(True, which="both")
    axs[2].set_xlim(
        0, max(x_len_all, x_len_att) - 1 if max(x_len_all, x_len_att) > 0 else 0
    )

    # 4) daily_side_counts/daily_side_dates plot
    # print("  Plotting daily_side_counts/daily_side_dates...")
    # print(f"daily_side_counts: {daily_side_counts}")
    axs[3].set_title("Side Performance (per day)")
    # We'll plot percent correct per side if possible
    legend_entries = []
    lines = []
    # Handle daily_side_counts as a 2xN array (rows: left, right; cols: days)
    if daily_side_counts is not None:
        # Accept if already an array; handle dict legacy format for compatibility
        if isinstance(daily_side_counts, dict):
            # Try conversion: expect dict with "attempts" as 2xN
            left_attempts = np.array(
                daily_side_counts.get("left", {}).get("attempts", [])
            )
            right_attempts = np.array(
                daily_side_counts.get("right", {}).get("attempts", [])
            )
            attempts_arr = np.vstack([left_attempts, right_attempts])
        else:
            # Fallback: assume it's already a tuple or array (corrects, attempts)
            # _, attempts_arr = daily_side_counts
            # Fix: Use daily_side_counts directly; expect shape (2, N)
            attempts_arr = np.array(daily_side_counts)

        # Sanity check: shape (2, Ndays)
        if attempts_arr.shape[0] != 2:
            print(
                "Warning: daily_side_counts attempts_arr unexpected shape:",
                attempts_arr.shape,
            )

        n_days = attempts_arr.shape[1]
        sides = ["left", "right"]
        colors = ["C0", "C1"]
        x = np.arange(n_days)

        width = 0.3
        offset = [-width / 2, width / 2]
        for i, side in enumerate(sides):
            axs[3].bar(
                x + offset[i],
                attempts_arr[i],
                width=width,
                alpha=0.7,
                label=f"{side} attempts",
                color=colors[i],
            )
        axs[3].set_ylabel("Attempts")
        axs[3].legend(loc="upper left")

        # Set proper x-ticks
        if daily_side_dates is not None and len(daily_side_dates) == n_days:
            xlabels = [str(d) for d in daily_side_dates]
            axs[3].set_xticks(np.arange(len(xlabels)))
            axs[3].set_xticklabels(xlabels, rotation=45, ha="right")

        axs[3].grid(True, which="both")
    else:
        axs[3].text(
            0.5,
            0.5,
            "No daily_side_counts data",
            ha="center",
            va="center",
            transform=axs[3].transAxes,
        )
    axs[3].set_xlabel("Day index (or date if given)")
    axs[3].set_xlim(left=0)
    plt.tight_layout()
    print("  Plotting complete.")
    print("--------------------------------------")
    return fig


def plot_tmaze_performance_arrays(
    all_performance_arr,
    custom_bins=None,
    bin_labels=None,
    kagen="None",
    all_attempts_arr=None,
    daily_side_counts=None,
    daily_side_dates=None,
):
    """

    Create a 4-panel summary figure for T-maze performance.

    STANDARDIZED INPUT SCHEMA
    -------------------------
    1) all_performance_arr : array-like, shape (bins, days) or (days,)
       - Raw performance (0..1) per bin per day.

    2) moving_averages_arr : either
           a) array-like, shape (bins, days) or (days,)
           b) dict {bin_name: 1D array-like of length days}
       - Moving-average (or smoothed) performance per bin.

    3) all_attempts_arr : array-like, optional, shape (bins, days) or (days,)
       - Number of attempts per bin per day.

    4) daily_side_counts : dict or None
       Expected strict schema:
           {
               "left":  {"attempts": array-like (days,), "correct": array-like (days,)},
               "right": {"attempts": array-like (days,), "correct": array-like (days,)}
           }

    5) daily_side_dates : sequence of length days or None
       - Used as x tick labels in subplot 4 (otherwise day index is used).

    SUBPLOTS
    --------
    1) Raw performance per bin over days (%)
    2) Moving-average performance per bin over days (%)
    3) Attempts per bin per day + total attempts
    4) Left/right attempts per day (bars) + percent correct (lines)

    Returns
    -------
    fig : matplotlib.figure.Figure
    """

    import numpy as np
    import matplotlib.pyplot as plt

    # ----------------------------------------------------------------------
    # Helper: robust 2D conversion (bins, days)
    # ----------------------------------------------------------------------
    def to_2d_array(arr, name):
        """
        Convert input to np.ndarray with shape (bins, days).

        Cases:
        - 1D -> (1, days)
        - 2D -> (bins, days)
        Raises if ndim > 2 or empty.
        """
        arr = np.asarray(arr)
        if arr.ndim == 1:
            arr = arr.reshape(1, -1)
        elif arr.ndim == 2:
            pass
        else:
            raise ValueError(
                f"{name} must be 1D or 2D array-like, got shape {arr.shape}"
            )
        if arr.size == 0:
            raise ValueError(f"{name} is empty.")
        return arr

    # ----------------------------------------------------------------------
    # Helper: centered moving average for 1D arrays (odd window)
    # ----------------------------------------------------------------------
    def moving_average_1d(a, n=3):
        """
        Compute centered moving average for a 1D array with an odd window n.

        - Uses 'valid' convolution then pads symmetrically on both sides.
        - If len(a) < n, returns NaNs of same length.
        """
        a = np.asarray(a, dtype=float)
        if a.ndim != 1:
            raise ValueError("moving_average_1d expects a 1D array.")
        if len(a) == 0:
            return a
        if n <= 1:
            return a.astype(float)
        if len(a) < n:
            return np.full_like(a, np.nan, dtype=float)

        kernel = np.ones(n, dtype=float) / n
        valid = np.convolve(a, kernel, mode="valid")

        diff = len(a) - len(valid)
        pad_left = diff // 2
        pad_right = diff - pad_left

        # Pad with edge values to keep length identical to input
        out = np.pad(valid, (pad_left, pad_right), mode="edge")
        return out

    # ----------------------------------------------------------------------
    # Helper: parse daily_side_counts into numpy arrays
    # ----------------------------------------------------------------------
    def parse_daily_side_counts(daily_side_counts, daily_side_dates):
        """
        Enforce and parse the strict schema for daily_side_counts.

        Returns:
            attempts_arr : np.ndarray, shape (2, days)    [0=left, 1=right]
            pct_correct  : np.ndarray, shape (2, days)    [0=left, 1=right]
            x_labels     : list of str, length days
        """
        if daily_side_counts is None:
            return None, None, None

        arr = np.asarray(daily_side_counts, dtype=float)

        if arr.ndim != 2 or arr.shape[0] != 2:
            raise ValueError(
                f"daily_side_counts must be a 2×N array of attempts. Got shape {arr.shape}"
            )

        left_attempts = arr[0]
        right_attempts = arr[1]

        # No correct data provided → set correct arrays to NaN
        left_correct = np.full_like(left_attempts, np.nan)
        right_correct = np.full_like(right_attempts, np.nan)

        # Basic length validation
        if not (
            len(left_attempts)
            == len(right_attempts)
            == len(left_correct)
            == len(right_correct)
        ):
            raise ValueError(
                "All daily_side_counts arrays (left/right, attempts/correct) must have the same length."
            )

        days = len(left_attempts)
        if days == 0:
            raise ValueError("daily_side_counts arrays are empty.")

        attempts_arr = np.vstack([left_attempts, right_attempts])

        # Avoid divide-by-zero; where attempts == 0 -> NaN percent correct
        left_pct = np.where(left_attempts > 0, left_correct / left_attempts, np.nan)
        right_pct = np.where(right_attempts > 0, right_correct / right_attempts, np.nan)
        pct_correct = np.vstack([left_pct, right_pct])

        # Build x labels
        if daily_side_dates is not None:
            if len(daily_side_dates) != days:
                raise ValueError(
                    "Length of daily_side_dates must match daily_side_counts day dimension."
                )
            x_labels = [str(d) for d in daily_side_dates]
        else:
            x_labels = [str(i) for i in range(days)]

        return attempts_arr, pct_correct, x_labels

    # ----------------------------------------------------------------------
    # 1) Convert and validate main performance array
    # ----------------------------------------------------------------------
    perf = to_2d_array(all_performance_arr, "all_performance_arr")
    n_bins, n_days = perf.shape

    # ----------------------------------------------------------------------
    # 2) Build / validate bin_labels
    # ----------------------------------------------------------------------
    if bin_labels is None:
        if custom_bins is not None:
            # Use edges to define labels like "0-2s", "2-4s", etc.
            custom_bins = list(custom_bins)
            if len(custom_bins) == n_bins + 1:
                bin_labels = [
                    f"{custom_bins[i]}-{custom_bins[i+1]}s" for i in range(n_bins)
                ]
            else:
                # Fallback to generic if custom_bins length doesn't match bins
                bin_labels = [f"Bin {i+1}" for i in range(n_bins)]
        else:
            bin_labels = [f"Bin {i+1}" for i in range(n_bins)]
    else:
        if len(bin_labels) != n_bins:
            raise ValueError(
                f"bin_labels length ({len(bin_labels)}) must match number of bins ({n_bins})."
            )

    # ----------------------------------------------------------------------
    # 3) Prepare moving_averages_arr in 2D form (bins, days)
    # ----------------------------------------------------------------------
    moving_averages_arr = (
        all_performance_arr  # shitty fix as the averaging was moved inside the function
    )
    if isinstance(moving_averages_arr, dict):
        # Dict case: {key: 1D array of length n_days}
        if len(moving_averages_arr) != n_bins:
            raise ValueError(
                "When moving_averages_arr is a dict, it must contain exactly one entry per bin."
            )
        mov_list = []
        for i, (k, arr) in enumerate(moving_averages_arr.items()):
            arr = np.asarray(arr, dtype=float)
            if arr.ndim != 1:
                raise ValueError(f"moving_averages_arr['{k}'] must be 1D.")
            if len(arr) != n_days:
                raise ValueError(
                    f"moving_averages_arr['{k}'] length ({len(arr)}) must match n_days ({n_days})."
                )
            mov_list.append(arr)
        mov = np.vstack(mov_list)
    else:
        # Array-like case
        mov = to_2d_array(moving_averages_arr, "moving_averages_arr")
        if mov.shape != perf.shape:
            raise ValueError(
                f"moving_averages_arr shape {mov.shape} must match all_performance_arr shape {perf.shape}."
            )

    # ----------------------------------------------------------------------
    # 4) Prepare attempts array if given
    # ----------------------------------------------------------------------
    attempts = None
    if all_attempts_arr is not None:
        attempts = to_2d_array(all_attempts_arr, "all_attempts_arr")
        if attempts.shape != perf.shape:
            raise ValueError(
                f"all_attempts_arr shape {attempts.shape} must match all_performance_arr shape {perf.shape}."
            )

    # ----------------------------------------------------------------------
    # 5) Parse daily_side_counts for subplot 4
    # ----------------------------------------------------------------------
    side_attempts, side_pct, side_xlabels = parse_daily_side_counts(
        daily_side_counts, daily_side_dates
    )

    # ----------------------------------------------------------------------
    # 6) Set up figure and common color cycle
    # ----------------------------------------------------------------------
    fig, axs = plt.subplots(4, 1, figsize=(12, 16), sharex=False)
    fig.suptitle(str(kagen), fontsize=14, weight="bold")

    # Use Matplotlib public API for default colors
    color_cycle = plt.rcParams.get("axes.prop_cycle", None)
    if color_cycle is not None:
        base_colors = color_cycle.by_key().get("color", [])
    else:
        base_colors = []
    if not base_colors:
        # Fallback if rcParams is unavailable or empty
        base_colors = [f"C{i}" for i in range(10)]

    # ----------------------------------------------------------------------
    # Subplot 1: Raw performance (%)
    # ----------------------------------------------------------------------
    ax0 = axs[0]
    ax0.set_title("Overall performance per bin")
    days_idx = np.arange(n_days)

    for i in range(n_bins):
        color = base_colors[i % len(base_colors)]
        ax0.plot(
            days_idx,
            perf[i] * 100.0,
            label=bin_labels[i],
            linewidth=1.5,
            color=color,
        )

    ax0.set_ylabel("Performance (%)")
    ax0.set_xlabel("Day index")
    ax0.set_ylim(0, 105)
    ax0.grid(True, which="both", linestyle=":")
    for yval in range(10, 100, 10):
        ax0.axhline(y=yval, color="gray", alpha=0.3, linewidth=1.0)
    ax0.set_xlim(0, n_days - 1)
    ax0.legend(title="Bins")

    # ----------------------------------------------------------------------
    # Subplot 2: Moving-average performance (%)
    # ----------------------------------------------------------------------
    ax1 = axs[1]
    ax1.set_title("T-Maze performance (3-day moving average)")

    for i in range(n_bins):
        color = base_colors[i % len(base_colors)]
        y_raw = mov[i] * 100.0
        y_smooth = moving_average_1d(mov[i], n=3) * 100.0

        # Raw (dashed)
        # ax1.plot(
        #     days_idx,
        #     y_raw,
        #     linestyle="--",
        #     linewidth=1.5,
        #     color=color,
        #     alpha=0.7,
        # )
        # Smoothed (solid)
        ax1.plot(
            days_idx,
            y_smooth,
            linestyle="-",
            linewidth=2.0,
            color=color,
            label=bin_labels[i] if i == 0 else None,  # don't duplicate labels
        )

    ax1.set_xlabel("Day index")
    ax1.set_ylabel("Performance (%)")
    ax1.set_ylim(0, 105)
    ax1.grid(True, which="both", linestyle=":")
    for yval in range(10, 100, 10):
        ax1.axhline(y=yval, color="gray", alpha=0.3, linestyle="--", linewidth=0.7)
    ax1.set_xlim(0, n_days - 1)
    # Single legend entry just to explain raw vs smooth (optional)
    # You could add a custom legend if you want clarity.

    # ----------------------------------------------------------------------
    # Subplot 3: Attempts per bin + total
    # ----------------------------------------------------------------------
    ax2 = axs[2]
    ax2.set_title("Attempts per bin (and total)")

    if attempts is not None:
        for i in range(n_bins):
            color = base_colors[i % len(base_colors)]
            ax2.plot(
                days_idx,
                attempts[i],
                label=bin_labels[i],
                linewidth=1.5,
                color=color,
            )
        total_attempts = attempts.sum(axis=0)
        ax2.plot(
            days_idx,
            total_attempts,
            color="black",
            linewidth=3.0,
            label="Total",
            zorder=4,
        )
        ax2.legend(title="Bins")
    else:
        ax2.text(
            0.5,
            0.5,
            "No attempt data",
            ha="center",
            va="center",
            transform=ax2.transAxes,
        )

    ax2.set_xlabel("Day index")
    ax2.set_ylabel("Attempts")
    ax2.grid(True, which="both", linestyle=":")
    ax2.set_xlim(0, n_days - 1)

    # ----------------------------------------------------------------------
    # Subplot 4: Side attempts & percent correct
    # ----------------------------------------------------------------------
    ax3 = axs[3]
    ax3.set_title("Side performance per day")

    if side_attempts is not None:
        n_days_side = side_attempts.shape[1]
        x = np.arange(n_days_side)
        width = 0.35

        # Bar: attempts
        left_color = base_colors[0 % len(base_colors)]
        right_color = base_colors[1 % len(base_colors)]

        ax3.bar(
            x - width / 2,
            side_attempts[0],
            width=width,
            alpha=0.7,
            label="Left attempts",
            color=left_color,
        )
        ax3.bar(
            x + width / 2,
            side_attempts[1],
            width=width,
            alpha=0.7,
            label="Right attempts",
            color=right_color,
        )

        ax3.set_ylabel("Attempts")

        # Twin axis: percent correct
        ax3b = ax3.twinx()
        ax3b.plot(
            x,
            side_pct[0] * 100.0,
            linestyle="-",
            marker="o",
            color=left_color,
            label="Left % correct",
        )
        ax3b.plot(
            x,
            side_pct[1] * 100.0,
            linestyle="-",
            marker="o",
            color=right_color,
            alpha=0.8,
            label="Right % correct",
        )
        ax3b.set_ylabel("Percent correct (%)")
        ax3b.set_ylim(0, 105)

        # X labels: dates or indices
        ax3.set_xticks(x)
        ax3.set_xticklabels(side_xlabels, rotation=45, ha="right")

        # Build combined legend for bars + lines
        handles1, labels1 = ax3.get_legend_handles_labels()
        handles2, labels2 = ax3b.get_legend_handles_labels()
        ax3.legend(handles1 + handles2, labels1 + labels2, loc="upper left")

        ax3.grid(True, which="both", linestyle=":")
        ax3.set_xlim(-0.5, n_days_side - 0.5)
    else:
        ax3.text(
            0.5,
            0.5,
            "No daily_side_counts data",
            ha="center",
            va="center",
            transform=ax3.transAxes,
        )
        ax3.set_xlabel("Day index (or date if given)")
        ax3.set_xlim(left=0)

    ax3.set_xlabel("Day index (or date)")

    # ----------------------------------------------------------------------
    # Layout & return
    # ----------------------------------------------------------------------
    plt.tight_layout(rect=[0, 0.03, 1, 0.97])  # leave space for suptitle
    return fig


def calculate_performance_vs_iti_bin(df, bins, date_range=None):
    """
    Calculate mean performance and sem in each ITI bin, for trials within a given date range.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame containing at least "dt between ROI" (float), "choice" (int or float), and "date" columns.
    bins : sequence of float
        The bin edges to digitize ITI ("dt between ROI").
    date_range : tuple or list, optional
        (start_date, end_date) in the same format as df["date"]. If not specified, use all dates in df.

    Returns
    -------
    result : dict
        Contains fields "bin_centers", "mean", "sem", "counts" (n in each bin)
    """
    use_df = df.copy()
    if date_range is not None:
        start, end = date_range
        # Try to type-convert df["date"], start, and end to comparable types (preferably pd.Timestamp)
        try:
            # If df["date"] looks like an int (e.g., yyyymmdd), convert to str first
            if np.issubdtype(use_df["date"].dtype, np.integer):
                use_df["date"] = use_df["date"].astype(str)
            use_df["date"] = pd.to_datetime(use_df["date"], errors="coerce")
            start_dt = pd.to_datetime(str(start), errors="coerce")
            end_dt = pd.to_datetime(str(end), errors="coerce")
            use_df = use_df[(use_df["date"] >= start_dt) & (use_df["date"] <= end_dt)]
        except Exception as e:
            print("Error in date range filtering:", e)
    # Remove NaN rows
    use_df = use_df.dropna(subset=["dt between ROI", "choice"])
    x = use_df["dt between ROI"].values.astype(float)
    y = use_df["choice"].values.astype(float)

    bin_indices = np.digitize(x, bins) - 1  # zero-indexed bins
    n_bins = len(bins) - 1
    bin_means = []
    bin_sems = []
    bin_counts = []
    bin_centers = [(bins[i] + bins[i + 1]) / 2.0 for i in range(n_bins)]

    for b in range(n_bins):
        perf_in_bin = y[bin_indices == b]
        if perf_in_bin.size > 0:
            mean_perf = np.nanmean(perf_in_bin)
            sem_perf = np.nanstd(perf_in_bin, ddof=1) / np.sqrt(perf_in_bin.size)
            count = perf_in_bin.size
        else:
            mean_perf = np.nan
            sem_perf = np.nan
            count = 0
        bin_means.append(mean_perf)
        bin_sems.append(sem_perf)
        bin_counts.append(count)

    result = {
        "bin_centers": np.array(bin_centers),
        "mean": np.array(bin_means),
        "sem": np.array(bin_sems),
        "counts": np.array(bin_counts),
    }
    return result


def smart_parse_date(val):
    # Handles both int (e.g., 20251006), str in "%Y%m%d", or ISO8601 str or pd.Timestamp.
    if isinstance(val, (int, np.integer)):
        return pd.to_datetime(str(val), format="%Y%m%d", errors="coerce")
    if isinstance(val, (pd.Timestamp, datetime.date, datetime.datetime)):
        return pd.to_datetime(val)
    try:
        # Try strict "%Y%m%d" first
        d = pd.to_datetime(str(val), format="%Y%m%d", errors="raise")
        return d
    except Exception:
        # Try flexible parsing (ISO8601 or other)
        d = pd.to_datetime(str(val), errors="raise")
        return d


def plot_tmaze_all_performance(outcomes, kage_data, startas, cols=None):
    """
    Plot TMAZE performance and attempts over days since start for all subjects (kages).

    Parameters
    ----------
    outcomes : dict
        Nested data structure containing TMAZE "all_performance" data.
    kage_data : dict
        Dictionary of {kage: DataFrame} holding raw behavioral data.
    startas : dict
        Mapping {kage: yyyymmdd_start_date}.
    cols : list, optional
        List of colors for the performance lines.
    """
    import matplotlib.pyplot as plt
    import numpy as np
    from datetime import datetime

    if cols is None:
        cols = ["red", "blue", "green", "magenta"]

    def yyyymmdd_to_date(yyyymmdd):
        s = str(int(yyyymmdd))
        return datetime(year=int(s[:4]), month=int(s[4:6]), day=int(s[6:8])).date()

    print("Starting plot generation...")
    print(f"TMAZE in outcomes: {'TMAZE' in outcomes}")

    if (
        "TMAZE" in outcomes
        and "full" in outcomes["TMAZE"]
        and "all_performance" in outcomes["TMAZE"]["full"]
    ):
        print("Found TMAZE data")

        if isinstance(outcomes["TMAZE"]["full"]["all_performance"], dict):
            all_perf_dict = outcomes["TMAZE"]["full"]["all_performance"]
        else:
            all_perf_dict = {kage: outcomes["TMAZE"]["full"]["all_performance"]}

        print(f"Processing {len(all_perf_dict)} kages: {list(all_perf_dict.keys())}")

        for this_kage in all_perf_dict:
            print(f"\nProcessing kage: {this_kage}")
            all_performance_arr = np.array(all_perf_dict[this_kage])
            print(f"Performance array shape: {all_performance_arr.shape}")

            # Data lookup and fallback attempts
            subset = None
            dates = None
            print(
                f"Checking kage_data... 'kage_data' in locals(): {'kage_data' in locals()}"
            )
            if this_kage in kage_data:
                print(f"Found {this_kage} in kage_data")
                subset = kage_data[this_kage]
                if hasattr(subset, "groupby"):  # likely a DataFrame
                    dates = np.unique(subset["date"].values)
                    print(f"Got {len(dates)} dates from kage_data")

            if subset is None or dates is None:
                print("Subset or dates is None, trying fallback...")
                try:
                    subset = locals()["subset"]
                    dates = np.unique(subset["date"].values)
                    print(f"Fallback succeeded: got {len(dates)} dates")
                except Exception as e:
                    print(f"Fallback failed: {e}")
                    print("CONTINUING (skipping this kage)")
                    continue

            print("Data preparation successful, creating plot...")

            # The bins used for TMAZE are:
            custom_bins = np.array([0, 120, 240, 360, 3600])
            bin_mids_sec = (custom_bins[:-1] + custom_bins[1:]) / 2
            bin_mids_min = bin_mids_sec / 60

            start_date_val = startas.get(this_kage, None)
            if start_date_val is not None and start_date_val > 0:
                start_date_obj = yyyymmdd_to_date(start_date_val)
                day_numbers = np.array(
                    [(yyyymmdd_to_date(date) - start_date_obj).days for date in dates]
                )
            else:
                day_numbers = np.arange(len(dates))

            if all_performance_arr.shape[1] == len(day_numbers):
                x = day_numbers
            else:
                x = np.arange(all_performance_arr.shape[1])

            plt.figure(figsize=(12, 6))
            ax = plt.gca()

            for i, line in enumerate(all_performance_arr):
                color = cols[i % len(cols)]
                if i < len(bin_mids_min) - 1:
                    bin_mid_val = bin_mids_min[i]
                    label = f"Bin {i+1}: ~{bin_mid_val:.0f} min"
                elif i == len(bin_mids_min) - 1:
                    label = f"Bin {i+1}: over 10 min"
                else:
                    label = f"Bin {i+1}"
                line_percent = line * 100
                ax.plot(x, line_percent, label=label, color=color)
                ax.scatter(x, line_percent, s=20, color=color)

            # --- Add right axis for daily attempts as a black line ---
            if hasattr(subset, "groupby"):
                attempts_by_day = subset.groupby("date").size().to_dict()
                attempts_list = [attempts_by_day.get(date, 0) for date in dates]
            else:
                attempts_list = np.zeros_like(x)

            ax2 = ax.twinx()
            ax2.plot(
                x, attempts_list, color="black", linewidth=2, label="Attempts per day"
            )
            ax2.set_ylabel("Attempts per day", color="black")
            ax2.tick_params(axis="y", labelcolor="black")

            h1, l1 = ax.get_legend_handles_labels()
            h2, l2 = ax2.get_legend_handles_labels()
            ax2.legend(h1 + h2, l1 + l2, loc="upper right")

            ax.set_xlabel("Day number since start")
            ax.set_ylabel("Performance (%)")
            ax.set_title(
                f"TMAZE All Performance by Days Since Start (4 Bins)\n{this_kage}\n"
                f"(Bin labels by bin center in minutes, last bin: over 10 min)"
            )
            ax.set_ylim(0, 100)
            ax.set_yticks(np.arange(0, 101, 10))
            for y in range(10, 100, 10):
                ax.axhline(y, color="gray", linestyle="--", linewidth=0.5, alpha=0.5)

            plt.tight_layout()
            plt.show()


def get_start_end_AC_task(homef, kgs):
    """
    Load one AC task interval per mouse from:

        <homef>/**/report_output/tasks/kage*_tasks.csv

    Exact duplicate rows and duplicate AC intervals are removed.
    If several distinct AC intervals remain, the longest valid AC interval is used.
    """
    homef = Path(homef)

    task_files = sorted(
        set(homef.rglob("report_output/tasks/kage*_tasks.csv"))
        | set(homef.rglob("report_output/tasks/Kage*_tasks.csv"))
    )

    if not task_files:
        raise FileNotFoundError(
            f"No kage*_tasks.csv files found under **/report_output/tasks inside:\n{homef}"
        )

    print(f"Found {len(task_files)} task CSV files.")

    def normalise_kage(value):
        match = re.search(r"\d+", str(value))
        return f"kage{int(match.group())}" if match else str(value).lower()

    files_by_kage = {}
    for file_path in task_files:
        kage_name = normalise_kage(file_path.stem.replace("_tasks", ""))
        files_by_kage.setdefault(kage_name, []).append(file_path)

    startas = {}
    galas = {}
    combined_df = {}

    for requested_kage in kgs:
        kage = normalise_kage(requested_kage)
        paths = files_by_kage.get(kage, [])

        if not paths:
            print(f"{kage}: no task CSV found.")
            continue

        frames = []
        for file_path in paths:
            try:
                temp = pd.read_csv(file_path)
                temp["source_file"] = str(file_path)
                frames.append(temp)
            except Exception as exc:
                print(f"{kage}: failed to read {file_path}: {exc}")

        if not frames:
            continue

        tasks = pd.concat(frames, ignore_index=True, sort=False)
        tasks = tasks.drop_duplicates().reset_index(drop=True)

        # Normalise column names while retaining the original table columns.
        col_lookup = {
            str(col).strip().lower().replace(" ", "_"): col
            for col in tasks.columns
        }

        event_col = next(
            (col_lookup[x] for x in ["event", "task", "task_name", "name"]
             if x in col_lookup),
            None,
        )
        start_col = next(
            (col_lookup[x] for x in
             ["datetime_start", "start_datetime", "start_time", "start"]
             if x in col_lookup),
            None,
        )
        end_col = next(
            (col_lookup[x] for x in
             ["datetime_end", "end_datetime", "end_time", "end"]
             if x in col_lookup),
            None,
        )

        if event_col is None or start_col is None:
            print(f"{kage}: required columns not found. Columns: {list(tasks.columns)}")
            continue

        tasks["datetime_start_clean"] = pd.to_datetime(
            tasks[start_col], errors="coerce"
        )

        if end_col is not None:
            tasks["datetime_end_clean"] = pd.to_datetime(
                tasks[end_col], errors="coerce"
            )
        else:
            tasks["datetime_end_clean"] = pd.NaT

        # Treat 1969/1970 sentinel end times as missing.
        tasks.loc[
            tasks["datetime_end_clean"] < pd.Timestamp("1980-01-01"),
            "datetime_end_clean",
        ] = pd.NaT

        tasks["event_clean"] = (
            tasks[event_col].astype(str).str.strip().str.upper()
        )
        tasks = tasks.sort_values("datetime_start_clean").reset_index(drop=True)

        # If an AC row is open-ended, close it at the next task start where possible.
        next_start = tasks["datetime_start_clean"].shift(-1)
        tasks["datetime_end_clean"] = tasks["datetime_end_clean"].fillna(next_start)

        ac = tasks[tasks["event_clean"] == "AC"].copy()
        ac = ac.dropna(subset=["datetime_start_clean", "datetime_end_clean"])
        ac["duration_seconds"] = (
            ac["datetime_end_clean"] - ac["datetime_start_clean"]
        ).dt.total_seconds()
        ac = ac[ac["duration_seconds"] > 0]
        ac = ac.drop_duplicates(
            subset=["datetime_start_clean", "datetime_end_clean"]
        )

        if ac.empty:
            print(f"{kage}: no valid closed AC interval found.")
            combined_df[kage] = tasks
            continue

        selected = ac.loc[ac["duration_seconds"].idxmax()]

        if len(ac) > 1:
            print(
                f"{kage}: {len(ac)} distinct AC intervals found; "
                "using the longest interval."
            )

        start_dt = selected["datetime_start_clean"]
        end_dt = selected["datetime_end_clean"]

        startas[kage] = int(start_dt.strftime("%Y%m%d"))
        galas[kage] = int(end_dt.strftime("%Y%m%d"))

        # Keep a clean table for the PDF page.
        table_cols = [event_col, start_col]
        if end_col is not None:
            table_cols.append(end_col)
        combined_df[kage] = tasks[table_cols].copy()

        print(f"{kage}: AC {start_dt} -> {end_dt}")

    return startas, galas, combined_df

def validate_file_path(file_path: str) -> bool:
    """Check if file exists and is readable."""
    return os.path.exists(file_path) and os.access(file_path, os.R_OK)


def validate_date_range(start_date: float, end_date: float) -> bool:
    """Validate that date range is logical."""
    return start_date <= end_date if start_date > 0 and end_date > 0 else True


def validate_dataframe(df: pd.DataFrame, required_columns: List[str]) -> bool:
    """Check if dataframe has required columns and is not empty."""
    return all(col in df.columns for col in required_columns) and not df.empty


def safe_division(numerator: float, denominator: float) -> float:
    """Safely perform division with zero check."""
    return numerator / denominator if denominator != 0 else 0.0


def bin_data(data, bins):
    """
    Bin the passed `data` into defined `bins`.

    Parameters
    ----------
    data : pandas.Series
        The pandas.Series to be binned (has to be one-dimensional, i.e. 1 column of your pandas.DataFrame).

    bins : numpy.array
        A numpy.array of bin delimiters (times separating the bins) in seconds; i.e. if you want to bin your data
        onto values between 0-2, 2-5 and 5-10, `bins=[0, 2, 5, 10]`.

    Returns
    -------
    binned_data : numpy.array of int
        A numpy.array of counts found in each separate bin.


    """
    binned_data = (
        data.groupby(pd.cut(data, bins=bins), observed=False).size().to_numpy()
    )
    return binned_data


# def bin_data_overlapping_windows(AC_subset):
#     """
#     Bin data using overlapping time windows instead of fixed bins.

#     Parameters
#     ----------
#     AC_subset : pandas.DataFrame
#         DataFrame containing at least "dt between ROI" (in seconds) and "choice" columns.

#     Returns
#     -------
#     result : dict
#         Contains:
#         - binned_data: numpy.array of counts in each window
#         - binned_data_filtered: numpy.array of correct choices (choice==1) in each window
#         - percentage: numpy.array of percentage correct for each window
#         - x_prct_smoo: float, mean of "dt between ROI"
#         - bin_labels: list of bin labels in minutes
#     """
#     # Define overlapping windows in seconds: [start, end] for each window
#     windows = [
#         (0, 120),      # 0-2 min
#         (60, 180),     # 1-3 min
#         (120, 300),    # 2-5 min
#         (180, 360),    # 3-6 min
#         (240, 420),    # 4-7 min
#         (300, 540),    # 5-9 min
#         (360, 600),    # 6-10 min
#         (420, 720),    # 7-12 min
#     ]

#     # Get the data
#     dt_data = AC_subset["dt between ROI"].values
#     choice_data = AC_subset["choice"].values

#     # Calculate binned_data: count all trials in each window
#     binned_data = np.zeros(len(windows))
#     for i, (start, end) in enumerate(windows):
#         mask = (dt_data >= start) & (dt_data < end)
#         binned_data[i] = np.sum(mask)

#     # Calculate binned_data_filtered: count correct choices (choice == 1) in each window
#     subset_filtered = AC_subset[AC_subset["choice"] == 1]
#     dt_filtered = subset_filtered["dt between ROI"].values

#     binned_data_filtered = np.zeros(len(windows))
#     for i, (start, end) in enumerate(windows):
#         mask = (dt_filtered >= start) & (dt_filtered < end)
#         binned_data_filtered[i] = np.sum(mask)

#     # Calculate percentage
#     percentage = np.zeros(len(windows))
#     for i in range(len(windows)):
#         if binned_data[i] > 0:
#             percentage[i] = (binned_data_filtered[i] / binned_data[i]) * 100
#         else:
#             percentage[i] = np.nan

#     # Set x_prct_smoo to mean of "dt between ROI"
#     x_prct_smoo = np.mean(AC_subset["dt between ROI"].values)

#     # Create bin labels in minutes (using midpoint of each window)
#     bin_labels = [(start + end) / 2 / 60 for start, end in windows]

#     return {
#         "binned_data": binned_data,
#         "binned_data_filtered": binned_data_filtered,
#         "percentage": percentage,
#         "x_prct_smoo": x_prct_smoo,
#         "bin_labels": bin_labels
#     }


def get_full_date_range(data_dates, N_gap_days=7):
    """
    Returns a full date range, filling the potential date holes between the passed `start_date` and `end_date`.

    Parameters
    ----------
    data_dates :

    N_gap_days int, optional
        Maximal gap in data [in units of days] to still show in the T-maze plot. This parameter limits the number of missing
        days to show in the final plot; some experiments were run with several months of missing data in between
        (animals completely out of the Kage) and this parameter makes the graphs easier to look at by artificially shortening
        the gap and thus bringing pre- and post-lesioned periods closer together. Set to `7` days by default.

    Returns
    -------
    full_date_range : numpy.array
        A numpy.array of all dates in integer YYYYMMDD format between passed `start_date` and `end_date` (inclusive).
    """
    from datetime import date, timedelta

    # Convert start and end dates into datetime format
    start_date = np.min(data_dates)
    end_date = np.max(data_dates)
    sdate = date(
        int(str(start_date)[:4]),
        int(str(start_date)[4:6]),
        int(str(start_date)[6:]),
    )
    edate = date(
        int(str(end_date)[:4]), int(str(end_date)[4:6]), int(str(end_date)[6:])
    )

    # Calculate the difference between end and start dates in datetime format
    delta = edate - sdate

    # Fill the gaps in dates
    full_date_range = []
    inserted_days = 0
    for i in range(delta.days + 1):
        day = int(str(sdate + timedelta(days=i)).replace("-", ""))
        if day in data_dates:
            full_date_range.append(day)
            inserted_days = 0  # if a day present in the data is inserted, reset the missing_date index;
            # You want to shorten only large uniform gaps (more than N_gap_days missing in a
            # single interval)
        else:
            full_date_range.append(day)
            inserted_days += 1

    return np.array(full_date_range)


def find_crossing_50(x, percentage):
    x = np.array(x)
    percentage = np.array(percentage)

    # Check if the first value is below 50%
    if percentage[0] < 50:
        return 0.3
        # return np.nan

    # if all above 50% - set time to 10 min
    if np.all(percentage >= 50):
        return 10

    # Find indices where the curve crosses 50%
    crossing_indices = np.where(np.diff(np.sign(percentage - 50)))[0]

    if len(crossing_indices) == 0:
        return None  # No crossing found

    # Take the first crossing index
    idx = crossing_indices[0]

    # Linear interpolation to find the precise crossing point
    x1, x2 = x[idx], x[idx + 1]
    y1, y2 = percentage[idx], percentage[idx + 1]

    # Calculate the crossing point using linear interpolation formula
    if y2 != y1:
        crossing_x = x1 + (50 - y1) * (x2 - x1) / (y2 - y1)
    else:
        crossing_x = (x1 + x2) / 2  # In case of flat line at 50%

    return crossing_x


# Define the exponential decay function
def exp_decay(x, A, k, C):
    x = np.array(x, dtype=np.float64)  # Ensure x is a numpy array of floats
    return A * np.exp(-k * x) + C


def get_tmaze_fit(bin_labels0, prctg):
    """Fit an exponential decay after removing empty/non-finite bins."""
    x = np.asarray(bin_labels0, dtype=float)
    y = np.asarray(prctg, dtype=float)

    valid = np.isfinite(x) & np.isfinite(y)
    x = x[valid]
    y = y[valid]

    if len(x) < 3:
        raise ValueError(
            f"At least 3 finite ITI bins are required for an exponential fit; "
            f"only {len(x)} were available."
        )

    A_guess = y[0] - y[-1]
    C_guess = y[-1]
    k_guess = 1.0
    p0 = [A_guess, k_guess, C_guess]

    params, _ = curve_fit(
        exp_decay,
        x,
        y,
        p0=p0,
        maxfev=20000,
    )
    A_fit, k_fit, C_fit = params

    predicted = exp_decay(x, A_fit, k_fit, C_fit)
    r2 = r2_score(y, predicted)

    return [A_fit, k_fit, C_fit, r2]


def calculate_sleep_in_bins(row, bins):
    sleep_in_bins = {key: 0 for key in bins}
    start, end = row["sleep_start"], row["sleep_end"]

    for bin_name, (bin_start, bin_end) in bins.items():
        overlap_start = max(start, bin_start)
        overlap_end = min(end, bin_end)
        if overlap_start < overlap_end:
            sleep_in_bins[bin_name] += overlap_end - overlap_start
    return sleep_in_bins


def performance_calculation(data, subset_dates, full_dates, custom_bins, delta_bins):
    """
    Calculate T-maze performance without a moving average.

    Parameters
    ----------
    data : pandas.DataFrame
        A pandas.DataFrame of T-maze results obtained from a T-maze results .csv file.

    subset_dates : numpy.array
        A numpy.array of all unique dates (in YYYYMMDD integer format) found in the passed `data`.

    full_dates : numpy.array
        A numpy.array of a full date range, which contains all dates within `subset_dates`. Used to detect missing dates.

    custom_bins : list of int
        A list of custom bin delimiters in seconds for binning data.

    delta_bins : int, optional
        The length of bins for uniform binning in seconds.

    Returns
    -------
    performance_data : numpy.array
        A numpy.array of T-maze performances for each date without moving averaging.
    """
    try:
        print("Checking input data to performance_calculation():")

        n_bins = len(custom_bins) - 1
        performance_data = np.empty((n_bins, 0))

        for i, date in enumerate(subset_dates):
            subset = data[data["date"] == date]

            # Bin data
            custom_binned_data = bin_data(
                data=subset["dt between ROI"], bins=custom_bins
            )
            # Filter for choice = 1
            subset_filtered = subset[subset["choice"] == 1]
            custom_binned_data_filtered = bin_data(
                data=subset_filtered["dt between ROI"], bins=custom_bins
            )

            daily_performance = np.empty(n_bins)
            for idx in range(n_bins):
                denom = custom_binned_data[idx]
                num = custom_binned_data_filtered[idx]
                if denom == 0:
                    daily_performance[idx] = np.nan
                else:
                    daily_performance[idx] = num / denom

            # Print detailed summary if any bin is NaN
            # Replace NaN elements with zero, leave non-NaN
            if np.any(np.isnan(daily_performance)):
                # print(f"Summary: NaN performance detected for date {date} (day index {i})")
                for idx in range(n_bins):
                    denom = custom_binned_data[idx]
                    num = custom_binned_data_filtered[idx]
                    # if np.isnan(daily_performance[idx]):
                    #     print(f"  Bin {idx}: NaN (num: {num}, denom: {denom})")
                    # else:
                    #     print(f"  Bin {idx}: {daily_performance[idx]:.3f} (num: {num}, denom: {denom})")
                # print("-" * 60)

            # Replace NaN values with zero before appending
            daily_performance_no_nan = np.nan_to_num(daily_performance, nan=0.0)
            performance_data = np.hstack(
                (performance_data, daily_performance_no_nan.reshape(-1, 1))
            )
        return performance_data

    except Exception as e:
        print(f"Performance calculation error: {e}")
        import traceback

        traceback.print_exc()
        return None


from datetime import datetime


def timestamp_to_date_and_seconds(microsecond_timestamp):
    # Convert microseconds to seconds (Unix timestamp)
    timestamp_seconds = microsecond_timestamp / 1_000_000

    # Create a datetime object from the timestamp
    dt = datetime.fromtimestamp(timestamp_seconds)

    # Get date in YYYYMMDD format
    date_str = dt.strftime("%Y%m%d")

    # Calculate seconds since start of day
    seconds_since_midnight = dt.hour * 3600 + dt.minute * 60 + dt.second

    return date_str, seconds_since_midnight


def count_rows_in_24h_range(df, start_date_yyyymmdd, start_seconds):
    # Convert start date to datetime
    start_date = datetime.strptime(str(start_date_yyyymmdd), "%Y%m%d")

    # Calculate start datetime (date + seconds)
    start_datetime = start_date + timedelta(seconds=start_seconds)

    # Calculate end datetime (24 hours later)
    end_datetime = start_datetime + timedelta(hours=24)

    # Convert all df timestamps to proper datetime objects
    df["full_datetime"] = df.apply(
        lambda row: datetime.strptime(str(row["date"]), "%Y%m%d")
        + timedelta(seconds=row["timestamp"]),
        axis=1,
    )

    # Filter rows within the time range
    mask = (df["full_datetime"] >= start_datetime) & (
        df["full_datetime"] <= end_datetime
    )
    filtered_df = df[mask]

    # Count left and right sides
    left_count = len(filtered_df[filtered_df["side"] == "left"])
    right_count = len(filtered_df[filtered_df["side"] == "right"])

    return left_count, right_count


def count_rows_in_2h_range(df, start_date_yyyymmdd, start_seconds):
    # Convert start date to datetime
    start_date = datetime.strptime(str(start_date_yyyymmdd), "%Y%m%d")

    # Calculate start datetime (date + seconds)
    start_datetime = start_date + timedelta(seconds=start_seconds)

    # Calculate end datetime (2 hours later)
    end_datetime = start_datetime + timedelta(hours=2)

    # Convert all df timestamps to proper datetime objects
    df["full_datetime"] = df.apply(
        lambda row: datetime.strptime(str(row["date"]), "%Y%m%d")
        + timedelta(seconds=row["timestamp"]),
        axis=1,
    )

    # Filter rows within the time range
    mask = (df["full_datetime"] >= start_datetime) & (
        df["full_datetime"] <= end_datetime
    )
    filtered_df = df[mask]

    # Count left and right sides
    left_count = len(filtered_df[filtered_df["side"] == "left"])
    right_count = len(filtered_df[filtered_df["side"] == "right"])

    return left_count, right_count

### Fix for the second report page

The `10+ min` bin has an infinite upper boundary. Its calculated midpoint was therefore `inf`, which caused Matplotlib to fail before saving the page containing:

- performance versus ITI
- raw exponential fit
- smoothed exponential fit
- task-event table

This version plots ITI bins at categorical x-positions and removes empty values before fitting.

## 4. Cohort metadata and group-level exports

The individual report is unchanged. This section adds:

- one long-format CSV containing each mouse's daily T-maze performance
- one long-format CSV containing each mouse's performance versus ITI
- cohort-level mean ± SEM plots
- genotype information loaded from `kage_descriptions.csv`

These exported CSV files are used by the separate master notebook.

### Genotype grouping

Each cohort is averaged separately by genotype.

Genotypes are loaded only from:

`<cohort>/DB/kage_descriptions.csv`

The `Kage` column identifies the mouse and the `Description` column provides the genotype. Mice missing from this file are skipped rather than placed into an `Unknown` group.

### Correction

The moving-average helper is now defined globally before the mouse loop. In the previous version it existed only inside the plotting function, so the CSV export code could not call it. This caused every mouse to fail before any records or PDF pages were saved.

### WT and NLGF only

This version filters the genotype metadata before processing any mice. APP1-em1BDS, NLF, and any other genotype are excluded from:

- individual PDF pages
- mouse-level CSV exports
- longitudinal cohort averages
- performance-versus-ITI cohort averages

The cohort mean ± SEM plots use a shaded SEM band rather than error bars.

In [9]:
def normalise_kage_id(value):
    """Convert 1, '1', 'Kage1', or 'kage1' to 'kage1'."""
    if pd.isna(value):
        return np.nan
    match = re.search(r"\d+", str(value))
    return f"kage{int(match.group())}" if match else str(value).strip().lower()


def infer_age_from_path(path):
    """Infer 6m, 12m, or 24m from the cohort folder name."""
    match = re.search(r"(?<!\d)(6|12|24)\s*m", str(path), flags=re.IGNORECASE)
    return f"{match.group(1)}m" if match else "Unknown"


def load_kage_metadata(homef):
    """
    Load genotype metadata strictly from:

        <cohort>/DB/kage_descriptions.csv

    Required columns:
        Kage
        Description
    """
    metadata_path = Path(homef) / "DB" / "kage_descriptions.csv"

    if not metadata_path.exists():
        raise FileNotFoundError(
            f"Genotype metadata not found: {metadata_path}"
        )

    metadata = pd.read_csv(metadata_path)
    metadata.columns = [str(column).strip() for column in metadata.columns]

    required_columns = {"Kage", "Description"}
    missing = required_columns - set(metadata.columns)

    if missing:
        raise ValueError(
            f"Missing required columns {sorted(missing)} in {metadata_path}. "
            f"Available columns: {list(metadata.columns)}"
        )

    metadata = metadata[["Kage", "Description"]].copy()
    metadata.columns = ["kage", "genotype"]

    metadata["kage"] = metadata["kage"].apply(normalise_kage_id)
    metadata["genotype"] = (
        metadata["genotype"]
        .astype(str)
        .str.strip()
    )

    metadata = metadata.dropna(subset=["kage", "genotype"])
    metadata = metadata.drop_duplicates(subset=["kage"], keep="first")

    print(f"Loaded genotype metadata: {metadata_path}")
    print(
        metadata["genotype"]
        .value_counts(dropna=False)
        .rename("n_mice")
    )

    return metadata


def sem(values):
    """SEM after removing missing values."""
    values = pd.to_numeric(pd.Series(values), errors="coerce").dropna()
    if len(values) <= 1:
        return np.nan
    return values.std(ddof=1) / np.sqrt(len(values))


# Only these two genotypes are included in reports and summaries.
GENO_ORDER = ["WT", "NLGF"]

GENO_COLORS = {
    "WT": "#34C6D3",
    "NLGF": "#F062B8",
}
ITI_ORDER = [
    "0–30 sec",
    "30 sec–2 min",
    "2–4 min",
    "4–6 min",
    "6–10 min",
    "10+ min",
]


def moving_average_1d(values, n=3):
    """
    Return a centred moving average with the same length as the input.

    Missing values are preserved where a complete window is unavailable.
    """
    values = np.asarray(values, dtype=float)

    if values.ndim != 1:
        raise ValueError("moving_average_1d expects a one-dimensional array.")

    if len(values) == 0:
        return values.copy()

    if n <= 1:
        return values.copy()

    if len(values) < n:
        return np.full(len(values), np.nan, dtype=float)

    smoothed = (
        pd.Series(values, dtype=float)
        .rolling(window=n, center=True, min_periods=n)
        .mean()
        .to_numpy()
    )

    return smoothed


### Updated learning criterion

The short-delay bins remain separate:

- 0–30 seconds
- 30 seconds–2 minutes

For each mouse, the notebook applies a 3-day moving average to each bin separately. The first criterion day is the start of the first 3-day period in which both curves are above 60% on all three days. Longer ITI bins remain plotted but are not used to define learning.

### Group learning analysis by genotype

This version creates:

1. Separate WT and NLGF learning-day panels.
2. Separate WT and NLGF cumulative learning curves.
3. A genotype-level criterion day, defined as the first day by which at least 50% of all mice in that genotype have reached their individual criterion.
4. Coloured vertical criterion lines on every cohort longitudinal ITI-bin plot.

The individual criterion remains unchanged: both the 0–30 sec and 30 sec–2 min 3-day moving-average curves must exceed 60% for 3 consecutive days.

In [10]:
import os

output_dir = os.path.join(homef, "report_output")
os.makedirs(output_dir, exist_ok=True)
pdf_path = os.path.join(output_dir, "tmaze_performance_plots.pdf")
print(f"Saving report to: {pdf_path}")
pdf = PdfPages(pdf_path)
pages_saved = 0

# ITI bins
custom_bins = np.array([0, 30, 120, 240, 360, 600, np.inf])
custom_bin_labels = [
    "0–30 sec",
    "30 sec–2 min",
    "2–4 min",
    "4–6 min",
    "6–10 min",
    "10+ min",
]

# Load the AC periods from the task CSV files once.
startas, galas, df_output = get_start_end_AC_task(homef, kgs)

# Learning criterion:
# first run of 3 consecutive days where BOTH separate short-delay curves
# (0-30 sec and 30 sec-2 min) are above 60% on the 3-day moving average.
learning_records = []

metadata = load_kage_metadata(homef)

# Remove every genotype except WT and NLGF before the mouse loop.
metadata = metadata[
    metadata["genotype"].isin(GENO_ORDER)
].copy()

genotype_lookup = dict(zip(metadata["kage"], metadata["genotype"]))

print("\nOnly these genotypes will be analysed:")
print(metadata["genotype"].value_counts())

missing_metadata_kages = [
    normalise_kage_id(kage)
    for kage in kgs
    if normalise_kage_id(kage) not in genotype_lookup
]

if missing_metadata_kages:
    print(
        "WARNING: These kages are missing from DB/kage_descriptions.csv "
        "and will be skipped:"
    )
    print(missing_metadata_kages)

cohort_name = Path(homef).name
cohort_age = infer_age_from_path(homef)

# Long-format rows used for cohort averages and the master notebook.
longitudinal_records = []
iti_records = []

# Set up list to collect exp fit parameters (kage, all params, method)
exp_fit_records = []

try:
    for kage in kgs:
        fig = None
        try:
            print(f"Processing {kage}")
            # minimal filtering and prep. returns all the dates
            loaded = process_tmaze_file(homef, kage)
            if loaded is None:
                continue
            df, dcopy = loaded

            if kage not in startas or kage not in galas:
                print(f"Skipping {kage}: no valid AC task period.")
                continue

            # Restrict every plot and calculation to the AC task period only.
            dcopy = dcopy.copy()
            dcopy["date_parsed"] = pd.to_datetime(
                dcopy["date"].astype(str), errors="coerce"
            )
            ac_start = pd.to_datetime(str(startas[kage]), format="%Y%m%d")
            ac_end = pd.to_datetime(str(galas[kage]), format="%Y%m%d")
            dcopy = dcopy[
                (dcopy["date_parsed"] >= ac_start)
                & (dcopy["date_parsed"] <= ac_end)
            ].copy()

            if dcopy.empty:
                print(f"Skipping {kage}: no T-maze rows inside the AC period.")
                continue

            dates = np.sort(dcopy["date"].dropna().unique())
            n_bins = len(custom_bins) - 1  # Number of bins for ITI
            n_days = len(dates)  # Number of unique days in the session

            perf = performance_calculation(
                dcopy, dates, dates, custom_bins, delta_bins="fuck off"
            )

            if perf is None:
                continue

            normalised_kage = normalise_kage_id(kage)

            if normalised_kage not in genotype_lookup:
                print(
                    f"Skipping {kage}: genotype not found in "
                    "DB/kage_descriptions.csv"
                )
                continue

            genotype = genotype_lookup[normalised_kage]

            if genotype not in GENO_ORDER:
                print(f"Skipping {kage}: genotype {genotype} is not WT or NLGF.")
                continue

            # Save one row per mouse × day × ITI bin.
            for bin_index, iti_label in enumerate(custom_bin_labels):
                raw_values = np.asarray(perf[bin_index], dtype=float) * 100
                moving_values = moving_average_1d(raw_values, n=3)

                for day_index, date_value in enumerate(dates):
                    longitudinal_records.append({
                        "mouse": normalise_kage_id(kage),
                        "genotype": genotype,
                        "age": cohort_age,
                        "cohort": cohort_name,
                        "day_index": day_index,
                        "date": date_value,
                        "ITI_bin": iti_label,
                        "performance_raw_percent": raw_values[day_index],
                        "performance_3day_percent": moving_values[day_index],
                    })

            # Build attempts array that show how many attempts occurred in each ITI bin for each day
            bin_idxs = (
                np.digitize(dcopy["dt between ROI"].values, custom_bins) - 1
            )  # Bin assignment for each attempt
            attempts_arr = np.zeros(
                (n_bins, n_days), dtype=int
            )  # Initialize attempt count array
            for i, dt in enumerate(dates):  # For each day
                mask = dcopy["date"].values == dt  # Mask for rows on this day
                this_bins = bin_idxs[mask]  # Bin indices for only this day's attempts
                for b in range(n_bins):  # For each ITI bin
                    attempts_arr[b, i] = np.sum(
                        this_bins == b
                    )  # Count attempts for this bin & day

            # Side counts
            side_pivot = (
                dcopy.groupby(["date", "side"])
                .size()
                .unstack(fill_value=0)
                .sort_index()
            )
            side_dates = side_pivot.index.values
            for col in ["left", "right"]:
                if col not in side_pivot.columns:
                    side_pivot[col] = 0
            side_counts = side_pivot[["left", "right"]].values.T

            # ----------------------------------------------------------
            # Learning criterion
            # ----------------------------------------------------------
            # Keep the 0-30 sec and 30 sec-2 min bins separate.
            # A mouse meets the learning criterion only when BOTH short-delay
            # bin curves are above 60% for the same 3 consecutive days.

            short_bin_0_30 = moving_average_1d(
                np.asarray(perf[0], dtype=float) * 100,
                n=2,
            )
            short_bin_30_120 = moving_average_1d(
                np.asarray(perf[1], dtype=float) * 100,
                n=2,
            )

            both_short_bins_above = (
                (short_bin_0_30 > 70)
                & (short_bin_30_120 > 70)
            )

            # Find the FIRST qualifying 3-day run.
            # The vertical line marks the first day of that run.
            learning_day_index = np.nan
            criterion_completion_day_index = np.nan

            for idx in range(len(both_short_bins_above) - 2):
                if np.all(both_short_bins_above[idx:idx + 3]):
                    learning_day_index = idx
                    criterion_completion_day_index = idx + 2
                    break

            learning_records.append({
                "kage": kage,
                "mouse": normalise_kage_id(kage),
                "genotype": genotype,
                "criterion_met": pd.notna(learning_day_index),
                "learning_day_index": learning_day_index,
                "learning_date": (
                    dates[int(learning_day_index)]
                    if pd.notna(learning_day_index)
                    else np.nan
                ),
                "criterion_completion_day_index": criterion_completion_day_index,
                "criterion_completion_date": (
                    dates[int(criterion_completion_day_index)]
                    if pd.notna(criterion_completion_day_index)
                    else np.nan
                ),
                "criterion_ITI_definition": (
                    "0-30 sec and 30 sec-2 min kept separate; "
                    "both required above threshold"
                ),
                "threshold_percent": 70,
                "moving_average_days": 3,
                "consecutive_days_required": 2,
                "n_AC_days": len(dates),
            })

            # Plot daily tmaze performance
            try:
                fig = plot_tmaze_performance_arrays(
                    perf,
                    custom_bins=custom_bins,
                    bin_labels=custom_bin_labels,
                    kagen=kage,
                    all_attempts_arr=attempts_arr,
                    daily_side_counts=side_counts,
                    daily_side_dates=side_dates,
                )
            except Exception as plot_error:
                print(
                    f"ERROR in plot_tmaze_performance_arrays for {kage}: {plot_error}"
                )
                traceback.print_exc()
                continue

            if fig is not None:
                # Learning criterion annotation on the 3-day moving-average panel.
                learning_ax = fig.axes[1]
                learning_ax.axhline(
                    70,
                    color="black",
                    linestyle="--",
                    linewidth=1.2,
                    alpha=0.8,
                    label="Learning threshold (70%)",
                )

                if pd.notna(learning_day_index):
                    learning_day_index = int(learning_day_index)
                    learning_ax.axvline(
                        learning_day_index,
                        color="black",
                        linestyle=":",
                        linewidth=1.5,
                        label=(
                            f"First criterion day: {learning_day_index} "
                            "(both ≤2-min bins)"
                        ),
                    )
                    learning_ax.axvspan(
                        learning_day_index,
                        len(dates) - 1,
                        color="gray",
                        alpha=0.10,
                    )

                learning_ax.legend(fontsize=8, loc="best")

                try:
                    pdf.savefig(fig, bbox_inches="tight", dpi=150)
                    plt.close(fig)
                    pages_saved += 1
                except Exception as save_error:
                    print(f"Error saving figure for {kage}: {save_error}")
                    traceback.print_exc()
                    try:
                        plt.close(fig)
                    except:
                        pass
            else:
                print(f"✗ Warning: No figure returned for {kage}")

            # Performance vs ITI bin + exponential fits + table on a single page
            try:
                import datetime

                # Convert start and end date to integers if necessary, then to datetime for window calculation
                start_val = startas[kage]
                end_val = galas[kage]
                if not isinstance(
                    start_val, (pd.Timestamp, datetime.date, datetime.datetime)
                ):
                    start_val = int(start_val)
                    start_date = pd.to_datetime(str(start_val), format="%Y%m%d")
                else:
                    start_date = pd.to_datetime(start_val)
                if not isinstance(
                    end_val, (pd.Timestamp, datetime.date, datetime.datetime)
                ):
                    end_val = int(end_val)
                    end_date = pd.to_datetime(str(end_val), format="%Y%m%d")
                else:
                    end_date = pd.to_datetime(end_val)
                n_days = (end_date - start_date).days

                result_date_ranges = [startas[kage], galas[kage]]
                print(
                    f"n_days in t-maze: {n_days}. (from {startas[kage]} to {galas[kage]})"
                )

                date_range = result_date_ranges
                results = calculate_performance_vs_iti_bin(
                    df=dcopy, bins=custom_bins, date_range=date_range
                )

                # Create combined figure with subplots
                fig_combo = plt.figure(figsize=(10, 14))
                gs = fig_combo.add_gridspec(3, 2, height_ratios=[1.0, 1.0, 1.2])

                # Top: Performance vs ITI
                ax_bin = fig_combo.add_subplot(gs[0, :])
                mean_100 = results["mean"] * 100
                sem_100 = results["sem"] * 100

                # Save one row per mouse × ITI bin. Group-level SEM is
                # recalculated across mice later; trial-level SEM is retained
                # here only as a QC field.
                for iti_index, iti_label in enumerate(custom_bin_labels):
                    iti_records.append({
                        "mouse": normalise_kage_id(kage),
                        "genotype": genotype,
                        "age": cohort_age,
                        "cohort": cohort_name,
                        "ITI_bin": iti_label,
                        "performance_percent": mean_100[iti_index],
                        "within_mouse_trial_sem_percent": sem_100[iti_index],
                        "n_trials": results["counts"][iti_index],
                    })

                # Use categorical x positions. The final 10+ bin has an
                # infinite upper edge, so its numeric midpoint is infinite and
                # cannot be passed to Matplotlib as an x-coordinate.
                iti_x = np.arange(len(custom_bin_labels))

                ax_bin.errorbar(
                    iti_x,
                    mean_100,
                    yerr=sem_100,
                    fmt="o-",
                    color="C0",
                    ecolor="gray",
                    elinewidth=2,
                    capsize=4,
                )
                ax_bin.set_xlabel("Inter-Trial Interval (ITI)", fontsize=12)
                ax_bin.set_ylabel("Mean Performance (%)", fontsize=12)
                ax_bin.set_title(
                    f"Performance vs ITI: {kage}", fontsize=14, fontweight="bold"
                )
                ax_bin.set_xticks(iti_x)
                ax_bin.set_xticklabels(custom_bin_labels, rotation=30, ha="right")
                ax_bin.set_ylim(0, 105)
                ax_bin.grid(True, axis="y", alpha=0.4)
                for y in range(0, 101, 10):
                    ax_bin.axhline(
                        y=y,
                        color="gray",
                        linestyle="--",
                        linewidth=0.7,
                        alpha=0.3,
                        zorder=0,
                    )

                binsz = custom_bins

                # Use smart_parse_date to flexibly parse the date, supporting both "%Y%m%d" and ISO8601 formats
                date0 = smart_parse_date(date_range[0])
                date1 = smart_parse_date(date_range[1])
                AC_subset = df.copy().dropna()
                AC_subset["date_ts"] = AC_subset["date"].apply(smart_parse_date)
                start_dt = smart_parse_date(startas[kage])
                end_dt = smart_parse_date(galas[kage])
                mask = (AC_subset["date_ts"] >= start_dt) & (
                    AC_subset["date_ts"] <= end_dt
                )
                AC_subset = AC_subset[mask].copy()
                AC_subset_dates = np.unique(AC_subset["date"].values)
                print(
                    f"AC_subset_dates:  first: {np.sort(AC_subset_dates)[0] if len(AC_subset_dates) > 0 else 'N/A'}, last: {np.sort(AC_subset_dates)[-1] if len(AC_subset_dates) > 0 else 'N/A'}"
                )
                binned_data = bin_data(data=AC_subset["dt between ROI"], bins=binsz)
                subset_filtered = AC_subset[AC_subset["choice"] == 1]
                binned_data_filtered = bin_data(
                    data=subset_filtered["dt between ROI"], bins=binsz
                )
                percentage = np.divide(
                    binned_data_filtered,
                    binned_data,
                    out=np.full_like(binned_data, np.nan, dtype=float),
                    where=binned_data > 0,
                ) * 100
                # bin_labels = [xx + 75 for xx in binsz[:-1]] + [1200, 3600]
                bin_labels = [
                    (binsz[i] + binsz[i + 1]) / 2 for i in range(len(binsz) - 1)
                ]
                bin_labels = [xx / 60 for xx in bin_labels]
                mem_span = find_crossing_50(bin_labels, percentage)
                finite_fit_bins = len(custom_bins) - 2
                prctg = percentage[:finite_fit_bins]
                bin_labels0 = bin_labels[:finite_fit_bins]

                # --- RAW binned fit (left middle) ---
                ax_fit = fig_combo.add_subplot(gs[1, 0])
                try:
                    params = get_tmaze_fit(bin_labels0, prctg)
                    print(f"EXPFIT: {kage}: {prctg}; {bin_labels0}; OUT: {params}")
                    A_fit, k_fit, C_fit, r2 = params
                    # For later, add fit record
                    tau_exp = 1 / k_fit if k_fit != 0 else np.nan
                    exp_fit_records.append(
                        {
                            "kage": kage,
                            "method": "raw_binned",
                            "A": A_fit,
                            "k": k_fit,
                            "C": C_fit,
                            "r2": r2,
                            "tau_exp_min": tau_exp,
                            "tau_50_min": mem_span,
                        }
                    )
                    raw_x = np.asarray(bin_labels0, dtype=float)
                    raw_y = np.asarray(prctg, dtype=float)
                    raw_valid = np.isfinite(raw_x) & np.isfinite(raw_y)
                    raw_x = raw_x[raw_valid]
                    raw_y = raw_y[raw_valid]

                    x_fit = np.linspace(np.min(raw_x), np.max(raw_x), 200)
                    y_fit = exp_decay(x_fit, A_fit, k_fit, C_fit)
                    ax_fit.plot(raw_x, raw_y, "o", label="Data")
                    ax_fit.plot(
                        x_fit, y_fit, "-", color="firebrick", label="Exponential Fit"
                    )
                    ax_fit.set_xlabel("Median ITI (min)")
                    ax_fit.set_ylabel("% Correct")
                    ax_fit.set_title(f"Memory Decay Fit - {kage}")
                    ax_fit.legend(
                        [
                            f"Data\n"
                            + f"tau_exp = {(1/params[1]):.3f} min\n"  # tau from exp fit, param k = 1/tau in exp_decay
                            + f"tau_50 = {mem_span:.2f} min\n"  # mem_span is tau_50
                            + f"A = {params[0]:.2f}%\n"
                            + f"C = {params[2]:.2f}%\n"
                            + f"R2 = {r2:.2f}",
                            "Exponential Fit",
                        ]
                    )
                    ax_fit.grid(alpha=0.3)
                except Exception:
                    params = [0, 0, 0, 0]
                    print(
                        f"Error fitting data: {params = }. Exception: {traceback.format_exc()}"
                    )

                # --- Smoothed overlapping window fit (right middle) ---
                counts, correct_counts, percentages, bin_times_min, x_mean_time = (
                    bin_data_overlapping_windows(AC_subset)[k]
                    for k in (
                        "binned_data",
                        "binned_data_filtered",
                        "percentage",
                        "bin_labels",
                        "x_prct_smoo",
                    )
                )
                ax_fit2 = fig_combo.add_subplot(gs[1, 1])
                try:
                    params2 = get_tmaze_fit(bin_times_min, percentages)
                    A_fit2, k_fit2, C_fit2, r22 = params2
                    tau_exp_s = 1 / k_fit2 if k_fit2 != 0 else np.nan
                    exp_fit_records.append(
                        {
                            "kage": kage,
                            "method": "smoothed",
                            "A": A_fit2,
                            "k": k_fit2,
                            "C": C_fit2,
                            "r2": r22,
                            "tau_exp_min": tau_exp_s,
                            "tau_50_min": mem_span,
                        }
                    )
                    x_fit2 = np.linspace(
                        np.min(bin_times_min), np.max(bin_times_min), 200
                    )
                    y_fit2 = exp_decay(x_fit2, A_fit2, k_fit2, C_fit2)
                    ax_fit2.plot(bin_times_min, percentages, "o", label="smoothed Data")
                    ax_fit2.plot(
                        x_fit2,
                        y_fit2,
                        "-",
                        color="firebrick",
                        label="smoothed Exponential Fit",
                    )
                    ax_fit2.plot(
                        bin_labels0,
                        prctg,
                        "o",
                        color="gray",
                        markersize=4,
                        alpha=0.7,
                        label="raw Data",
                    )
                    ax_fit2.plot(
                        x_fit,
                        y_fit,
                        "-",
                        color="gray",
                        linewidth=1,
                        alpha=0.7,
                        label="raw Exponential Fit",
                    )
                    ax_fit2.set_xlabel("Median ITI (min)")
                    ax_fit2.set_ylabel("% Correct")
                    ax_fit2.set_title(f"Memory Decay Fit [smoothed] - {kage}")
                    ax_fit2.legend(
                        [
                            f"Smoothed data\n"
                            + f"tau_exp = {(1/params2[1]):.3f} min\n"  # tau from exp fit, param k = 1/tau in exp_decay
                            + f"tau_50 = {mem_span:.2f} min\n"  # mem_span is tau_50
                            + f"A = {params2[0]:.2f}%\n"
                            + f"C = {params2[2]:.2f}%"
                            + f"R2 = {r22:.2f}",
                            "Smoothed exponential Fit",
                            "Raw data",
                            "Raw exponential Fit",
                        ]
                    )
                    ax_fit2.grid(alpha=0.3)
                except Exception:
                    params2 = [0, 0, 0, 0]
                    print(
                        f"Error fitting data: {params2 = }. Exception: {traceback.format_exc()}"
                    )

                # Table of summary stats at the bottom
                ax_table = fig_combo.add_subplot(gs[2, :])
                ax_table.axis("off")
                ax_table.set_title(
                    f"Kage: {kage}", fontsize=14, fontweight="bold", pad=20
                )
                tbl = ax_table.table(
                    cellText=df_output[kage].values,
                    colLabels=df_output[kage].columns,
                    loc="center",
                    cellLoc="center",
                )
                tbl.auto_set_font_size(False)
                tbl.set_fontsize(8)
                tbl.scale(1.2, 1.2)

                fig_combo.tight_layout(rect=[0, 0, 1, 0.97])
                pdf.savefig(fig_combo, bbox_inches="tight", dpi=150)
                plt.close(fig_combo)
                pages_saved += 1
            except Exception as err:
                print(f"Error saving combined ITI/fit/table page for {kage}: {err}")
                import traceback

                traceback.print_exc()
            print(f"Finished {kage}")
        except Exception as e:
            print(f"Error with {kage}: {e}")
            traceback.print_exc()
            if fig is not None:
                try:
                    plt.close(fig)
                except:
                    pass
            continue
finally:

    # Add genotype-separated cohort-level learning summaries.
    try:
        learning_df = pd.DataFrame(learning_records)

        if not learning_df.empty:
            learning_df = learning_df[
                learning_df["genotype"].isin(GENO_ORDER)
            ].copy()

            # ------------------------------------------------------
            # Group criterion day
            # ------------------------------------------------------
            # For the longitudinal group plots, the group criterion day is
            # the first AC task day by which at least 80% of ALL mice in that
            # genotype have reached their individual learning criterion.
            group_criterion_days = {}

            for genotype in GENO_ORDER:
                genotype_learning = learning_df[
                    learning_df["genotype"] == genotype
                ].copy()

                if genotype_learning.empty:
                    group_criterion_days[genotype] = np.nan
                    continue

                max_day = int(genotype_learning["n_AC_days"].max())
                criterion_day = np.nan

                for day in range(max_day):
                    percent_reached = 100 * np.mean(
                        genotype_learning["criterion_met"]
                        & (
                            genotype_learning["learning_day_index"]
                            <= day
                        )
                    )

                    if percent_reached >= 80:
                        criterion_day = day
                        break

                group_criterion_days[genotype] = criterion_day

            print("\nGroup learning criterion days:")
            for genotype, day in group_criterion_days.items():
                print(f"{genotype}: {day}")

            # ------------------------------------------------------
            # Figure 1: learning day by mouse, separated by genotype
            # ------------------------------------------------------
            fig_learning_days, axes_days = plt.subplots(
                1,
                len(GENO_ORDER),
                figsize=(12, 5),
                sharey=True,
            )
            axes_days = np.atleast_1d(axes_days)

            for ax, genotype in zip(axes_days, GENO_ORDER):
                genotype_df = learning_df[
                    learning_df["genotype"] == genotype
                ].copy()

                learned_df = genotype_df[
                    genotype_df["criterion_met"]
                ].sort_values("learning_day_index")

                if not learned_df.empty:
                    x = np.arange(len(learned_df))
                    ax.scatter(
                        x,
                        learned_df["learning_day_index"],
                        s=55,
                        color=GENO_COLORS.get(genotype),
                        label="Reached criterion",
                    )
                    ax.set_xticks(x)
                    ax.set_xticklabels(
                        learned_df["kage"],
                        rotation=60,
                        ha="right",
                    )
                else:
                    ax.text(
                        0.5,
                        0.5,
                        "No mice reached criterion",
                        ha="center",
                        va="center",
                        transform=ax.transAxes,
                    )

                n_total = len(genotype_df)
                n_learned = int(genotype_df["criterion_met"].sum())
                ax.set_title(
                    f"{genotype}\n{n_learned}/{n_total} mice reached criterion"
                )
                ax.set_xlabel("Mouse")
                ax.grid(axis="y", alpha=0.3)
                ax.spines["top"].set_visible(False)
                ax.spines["right"].set_visible(False)

            axes_days[0].set_ylabel("First criterion day index")
            fig_learning_days.suptitle(
                "Learning day by mouse and genotype\n"
                "Both short-delay bins >70% for 2 consecutive days",
                fontsize=14,
            )
            fig_learning_days.tight_layout()

            learning_days_path = os.path.join(
                output_dir,
                "tmaze_learning_days_by_genotype.pdf",
            )
            fig_learning_days.savefig(
                learning_days_path,
                bbox_inches="tight",
                transparent=True,
            )
            pdf.savefig(fig_learning_days, bbox_inches="tight", dpi=150)
            plt.close(fig_learning_days)
            pages_saved += 1

            # ------------------------------------------------------
            # Figure 2: cumulative learning curves, separate panels
            # ------------------------------------------------------
            fig_cumulative, axes_cumulative = plt.subplots(
                1,
                len(GENO_ORDER),
                figsize=(12, 5),
                sharex=False,
                sharey=True,
            )
            axes_cumulative = np.atleast_1d(axes_cumulative)

            for ax, genotype in zip(axes_cumulative, GENO_ORDER):
                genotype_df = learning_df[
                    learning_df["genotype"] == genotype
                ].copy()

                if genotype_df.empty:
                    ax.text(
                        0.5,
                        0.5,
                        "No mice available",
                        ha="center",
                        va="center",
                        transform=ax.transAxes,
                    )
                    continue

                max_day = int(genotype_df["n_AC_days"].max())
                day_values = np.arange(max_day)

                reached = [
                    100
                    * np.mean(
                        genotype_df["criterion_met"]
                        & (
                            genotype_df["learning_day_index"]
                            <= day
                        )
                    )
                    for day in day_values
                ]

                ax.step(
                    day_values,
                    reached,
                    where="post",
                    linewidth=2.5,
                    color=GENO_COLORS.get(genotype),
                )

                group_day = group_criterion_days.get(genotype, np.nan)
                if pd.notna(group_day):
                    ax.axvline(
                        group_day,
                        linestyle=":",
                        linewidth=1.8,
                        color=GENO_COLORS.get(genotype),
                        label=f"50% group criterion: day {int(group_day)}",
                    )

                ax.axhline(
                    50,
                    linestyle="--",
                    linewidth=1,
                    color="black",
                    alpha=0.6,
                )
                ax.set_ylim(0, 105)
                ax.set_xlabel("AC task day index")
                ax.set_title(genotype)
                ax.grid(alpha=0.3)
                ax.spines["top"].set_visible(False)
                ax.spines["right"].set_visible(False)
                if pd.notna(group_day):
                    ax.legend(frameon=False, fontsize=8)

            axes_cumulative[0].set_ylabel("Mice reaching criterion (%)")
            fig_cumulative.suptitle(
                "Cumulative learning curves by genotype",
                fontsize=14,
            )
            fig_cumulative.tight_layout()

            cumulative_path = os.path.join(
                output_dir,
                "tmaze_cumulative_learning_by_genotype.pdf",
            )
            fig_cumulative.savefig(
                cumulative_path,
                bbox_inches="tight",
                transparent=True,
            )
            pdf.savefig(fig_cumulative, bbox_inches="tight", dpi=150)
            plt.close(fig_cumulative)
            pages_saved += 1

            learning_csv_path = (
                os.path.splitext(pdf_path)[0]
                + "_learning_criterion.csv"
            )
            learning_df.to_csv(
                learning_csv_path,
                index=False,
            )
            print(
                f"Learning criterion results saved to: "
                f"{learning_csv_path}"
            )

            group_criterion_df = pd.DataFrame(
                {
                    "genotype": list(group_criterion_days.keys()),
                    "group_criterion_day_50_percent": list(
                        group_criterion_days.values()
                    ),
                }
            )
            group_criterion_df.to_csv(
                os.path.join(
                    output_dir,
                    "tmaze_group_learning_criterion_by_genotype.csv",
                ),
                index=False,
            )

    except Exception as learning_exc:
        print(
            f"ERROR creating learning criterion summary: "
            f"{learning_exc}"
        )
        traceback.print_exc()


    # Save individual-mouse data and create cohort-level mean ± SEM plots.
    try:
        longitudinal_df = pd.DataFrame(longitudinal_records)
        iti_df = pd.DataFrame(iti_records)

        if longitudinal_df.empty:
            print(
                "WARNING: No longitudinal mouse-level rows were created. "
                "Cohort longitudinal summaries will not be generated."
            )

        if iti_df.empty:
            print(
                "WARNING: No performance-vs-ITI mouse-level rows were created. "
                "Cohort ITI summaries will not be generated."
            )

        longitudinal_csv = os.path.join(
            output_dir,
            "tmaze_longitudinal_individual_mice.csv",
        )
        iti_csv = os.path.join(
            output_dir,
            "tmaze_performance_vs_ITI_individual_mice.csv",
        )

        longitudinal_df.to_csv(longitudinal_csv, index=False)
        iti_df.to_csv(iti_csv, index=False)

        print(f"Saved longitudinal mouse-level data: {longitudinal_csv}")
        print(f"Saved ITI mouse-level data: {iti_csv}")

        # ----------------------------------------------------------
        # Cohort longitudinal mean ± SEM
        # ----------------------------------------------------------
        if "genotype" in longitudinal_df.columns:
            longitudinal_valid = longitudinal_df[
                longitudinal_df["genotype"].isin(GENO_ORDER)
            ].copy()
        else:
            longitudinal_valid = pd.DataFrame()

        print("\nMouse counts used for cohort averages:")
        if not longitudinal_valid.empty:
            print(
                longitudinal_valid
                .groupby(["cohort", "genotype"])["mouse"]
                .nunique()
                .rename("n_mice")
            )

        if not longitudinal_valid.empty:
            summary_long = (
                longitudinal_valid
                .groupby(
                    ["genotype", "ITI_bin", "day_index"],
                    observed=True,
                )
                .agg(
                    mean_performance=("performance_3day_percent", "mean"),
                    sem_performance=("performance_3day_percent", sem),
                    n_mice=("mouse", "nunique"),
                )
                .reset_index()
            )

            summary_long.to_csv(
                os.path.join(
                    output_dir,
                    "tmaze_longitudinal_cohort_mean_SEM.csv",
                ),
                index=False,
            )

            present_bins = [
                label
                for label in ITI_ORDER
                if label in summary_long["ITI_bin"].unique()
            ]

            fig_long, axes = plt.subplots(
                len(present_bins),
                1,
                figsize=(9, max(4, 3.2 * len(present_bins))),
                sharex=True,
            )
            axes = np.atleast_1d(axes)

            for ax, iti_label in zip(axes, present_bins):
                bin_data = summary_long[
                    summary_long["ITI_bin"] == iti_label
                ]

                for genotype in GENO_ORDER:
                    group = bin_data[
                        bin_data["genotype"] == genotype
                    ].sort_values("day_index")

                    if group.empty:
                        continue

                    x = group["day_index"].to_numpy()
                    y = group["mean_performance"].to_numpy()
                    error = group["sem_performance"].fillna(0).to_numpy()

                    ax.plot(
                        x,
                        y,
                        marker="o",
                        markersize=3,
                        linewidth=2,
                        label=genotype,
                        color=GENO_COLORS.get(genotype),
                    )
                    ax.fill_between(
                        x,
                        y - error,
                        y + error,
                        alpha=0.18,
                        color=GENO_COLORS.get(genotype),
                    )

                ax.axhline(
                    70,
                    linestyle="--",
                    linewidth=1,
                    color="black",
                    alpha=0.7,
                    label="Learning threshold (70%)",
                )

                # Show the genotype-level learning criterion on each bin plot.
                # This is the first day by which >=50% of all mice in that
                # genotype have reached their individual criterion.
                for genotype in GENO_ORDER:
                    group_day = group_criterion_days.get(genotype, np.nan)

                    if pd.notna(group_day):
                        ax.axvline(
                            int(group_day),
                            linestyle=":",
                            linewidth=1.8,
                            color=GENO_COLORS.get(genotype),
                            alpha=0.95,
                            label=(
                                f"{genotype} group criterion "
                                f"(50%): day {int(group_day)}"
                            ),
                        )

                ax.set_ylim(0, 105)
                ax.set_ylabel("Performance (%)")
                ax.set_title(f"{iti_label} | 3-day moving average")
                ax.spines["top"].set_visible(False)
                ax.spines["right"].set_visible(False)

            axes[-1].set_xlabel("AC task day index")
            handles, labels = axes[0].get_legend_handles_labels()
            if handles:
                unique_legend = dict(zip(labels, handles))
                fig_long.legend(
                    unique_legend.values(),
                    unique_legend.keys(),
                    loc="upper center",
                    ncol=2,
                    frameon=False,
                )

            fig_long.suptitle(
                f"Longitudinal T-maze performance: {cohort_name}",
                y=1.01,
                fontsize=15,
            )
            fig_long.tight_layout()

            longitudinal_plot_path = os.path.join(
                output_dir,
                "tmaze_longitudinal_cohort_mean_SEM.pdf",
            )
            fig_long.savefig(
                longitudinal_plot_path,
                bbox_inches="tight",
                transparent=True,
            )
            pdf.savefig(fig_long, bbox_inches="tight", dpi=150)
            plt.close(fig_long)
            pages_saved += 1

        # ----------------------------------------------------------
        # Cohort performance versus ITI mean ± SEM
        # ----------------------------------------------------------
        if "genotype" in iti_df.columns:
            iti_valid = iti_df[
                iti_df["genotype"].isin(GENO_ORDER)
            ].copy()
        else:
            iti_valid = pd.DataFrame()

        if not iti_valid.empty:
            iti_valid["ITI_bin"] = pd.Categorical(
                iti_valid["ITI_bin"],
                categories=ITI_ORDER,
                ordered=True,
            )

            summary_iti = (
                iti_valid
                .groupby(
                    ["genotype", "ITI_bin"],
                    observed=True,
                )
                .agg(
                    mean_performance=("performance_percent", "mean"),
                    sem_performance=("performance_percent", sem),
                    n_mice=("mouse", "nunique"),
                )
                .reset_index()
                .sort_values(["genotype", "ITI_bin"])
            )

            summary_iti.to_csv(
                os.path.join(
                    output_dir,
                    "tmaze_performance_vs_ITI_cohort_mean_SEM.csv",
                ),
                index=False,
            )

            fig_iti, ax = plt.subplots(figsize=(8, 5))
            x = np.arange(len(ITI_ORDER))

            for genotype in GENO_ORDER:
                group = summary_iti[
                    summary_iti["genotype"] == genotype
                ].set_index("ITI_bin").reindex(ITI_ORDER)

                if group["mean_performance"].notna().sum() == 0:
                    continue

                y = group["mean_performance"].to_numpy(dtype=float)
                error = (
                    group["sem_performance"]
                    .fillna(0)
                    .to_numpy(dtype=float)
                )

                valid = np.isfinite(y)

                ax.plot(
                    x[valid],
                    y[valid],
                    marker="o",
                    markersize=5,
                    linewidth=2.2,
                    label=genotype,
                    color=GENO_COLORS.get(genotype),
                )

                ax.fill_between(
                    x[valid],
                    (y - error)[valid],
                    (y + error)[valid],
                    alpha=0.20,
                    color=GENO_COLORS.get(genotype),
                    linewidth=0,
                )

            ax.axhline(50, linestyle="--", linewidth=1, color="black")
            ax.set_xticks(x)
            ax.set_xticklabels(ITI_ORDER, rotation=30, ha="right")
            ax.set_ylim(0, 105)
            ax.set_ylabel("Mean performance (%)")
            ax.set_xlabel("Inter-trial interval")
            ax.set_title(
                f"Performance versus ITI: {cohort_name}\nmean ± SEM across mice"
            )
            ax.legend(frameon=False)
            ax.spines["top"].set_visible(False)
            ax.spines["right"].set_visible(False)
            fig_iti.tight_layout()

            iti_plot_path = os.path.join(
                output_dir,
                "tmaze_performance_vs_ITI_cohort_mean_SEM.pdf",
            )
            fig_iti.savefig(
                iti_plot_path,
                bbox_inches="tight",
                transparent=True,
            )
            pdf.savefig(fig_iti, bbox_inches="tight", dpi=150)
            plt.close(fig_iti)
            pages_saved += 1

    except Exception as cohort_summary_error:
        print(
            "ERROR saving cohort-level T-maze summaries: "
            f"{cohort_summary_error}"
        )
        traceback.print_exc()

    pdf.close()
    print(f"\n{'='*60}")
    print(f"PDF closed successfully. Total pages saved: {pages_saved}")
    print(f"{'='*60}")
    if pages_saved == 0:
        print("WARNING: No pages were saved to the PDF!")

    # -- Save exp_fit_records as csv in the same directory as the PDF --
    try:
        fit_df = pd.DataFrame(exp_fit_records)
        if len(fit_df) > 0:
            fit_csv_path = (
                os.path.splitext(pdf_path)[0] + "_exponential_fit_results.csv"
            )
            fit_df.to_csv(fit_csv_path, index=False)
            print(f"Exponential fit results saved to: {fit_csv_path}")
        else:
            print("WARNING: No exponential fit results to save!")
    except Exception as fit_save_exc:
        print(f"ERROR saving exponential fit data CSV: {fit_save_exc}")
        traceback.print_exc()

Saving report to: /Volumes/Aimi_SmrtKg/2024-05-may-jul_NLGF6m/report_output/tmaze_performance_plots.pdf
Found 87 task CSV files.
kage1: AC 2024-05-20 11:03:20 -> 2024-06-13 10:31:03
kage2: 2 distinct AC intervals found; using the longest interval.
kage2: AC 2024-05-20 11:03:26 -> 2024-06-13 10:31:06
kage3: AC 2024-05-20 11:03:29 -> 2024-06-13 10:31:08
kage4: 2 distinct AC intervals found; using the longest interval.
kage4: AC 2024-05-23 14:55:13 -> 2024-06-13 10:31:10
kage5: AC 2024-05-27 12:01:33 -> 2024-06-18 10:32:52
kage6: AC 2024-05-20 11:03:39 -> 2024-06-13 10:31:12
kage7: AC 2024-05-20 11:03:43 -> 2024-06-13 10:31:14
kage8: AC 2024-05-20 11:03:46 -> 2024-06-13 10:31:16
kage9: 2 distinct AC intervals found; using the longest interval.
kage9: AC 2024-05-30 14:33:37 -> 2024-06-13 10:31:18
kage10: 2 distinct AC intervals found; using the longest interval.
kage10: AC 2024-05-23 14:55:22 -> 2024-06-13 10:31:20
kage11: AC 2024-05-20 11:03:55 -> 2024-06-13 10:31:21
kage12: AC 2024-05-20